# Ecuación de ondas 1D: cuaderno de solución

Este cuaderno presenta una implementación completa del esquema explícito centrado para la ecuación de ondas en 1D.

`T` es el tiempo final de simulación (`t_final`).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def exact_solution(x, t, c=1.0):
    return np.sin(np.pi * x) * np.cos(c * np.pi * t)


def u0(x):
    return np.sin(np.pi * x)


def v0(x):
    return np.zeros_like(x)


def build_grid(L, T, N, sigma_obj, c=1.0):
    dx = L / N
    dt = sigma_obj * dx / c
    nt = max(1, int(np.ceil(T / dt)))
    dt = T / nt
    sigma = c * dt / dx
    x = np.linspace(0.0, L, N + 1)
    return x, dx, dt, nt, sigma


def first_step(u_prev, v_init, sigma, dt):
    u_curr = u_prev.copy()
    u_curr[1:-1] = (
        u_prev[1:-1]
        + dt * v_init[1:-1]
        + 0.5 * sigma**2 * (u_prev[2:] - 2.0 * u_prev[1:-1] + u_prev[:-2])
    )
    return u_curr


def wave_step(u_prev, u_curr, sigma):
    u_next = np.empty_like(u_curr)
    u_next[1:-1] = (
        2.0 * u_curr[1:-1]
        - u_prev[1:-1]
        + sigma**2 * (u_curr[2:] - 2.0 * u_curr[1:-1] + u_curr[:-2])
    )
    return u_next


def solve_wave(c=1.0, L=1.0, T=0.6, N=220, sigma_obj=0.9, store_every=5):
    x, dx, dt, nt, sigma = build_grid(L, T, N, sigma_obj, c=c)

    u_prev = u0(x)
    u_prev[0], u_prev[-1] = 0.0, 0.0

    v_init = v0(x)
    u_curr = first_step(u_prev, v_init, sigma, dt)
    u_curr[0], u_curr[-1] = 0.0, 0.0

    t_hist = [0.0, dt]
    U_hist = [u_prev.copy(), u_curr.copy()]

    for n in range(1, nt):
        u_next = wave_step(u_prev, u_curr, sigma)
        t_next = (n + 1) * dt
        u_next[0], u_next[-1] = 0.0, 0.0

        if (n + 1) % store_every == 0 or (n + 1) == nt:
            t_hist.append(t_next)
            U_hist.append(u_next.copy())

        u_prev, u_curr = u_curr, u_next

    t_final = nt * dt
    return x, u_curr, t_final, sigma, np.array(t_hist), np.array(U_hist)


In [ ]:
# Parámetros
a = dict(c=1.0, L=1.0, T=0.6, N=220, sigma_obj=0.9, store_every=5)

x, u_num, t_final, sigma, t_hist, U_hist = solve_wave(**a)
u_ex = exact_solution(x, t_final, c=a['c'])

err_l2 = np.sqrt(np.mean((u_num - u_ex) ** 2))

print(f"t_final={t_final:.6f}")
print(f"sigma={sigma:.6f}")
print(f"error L2={err_l2:.3e}")


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(x, u_num, lw=2, label='Numérica')
plt.plot(x, u_ex, '--', lw=2, label='Exacta')
plt.title('Ecuación de ondas 1D en t_final')
plt.xlabel('x')
plt.ylabel('u(x,t_final)')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
line_num, = ax.plot([], [], lw=2, label='Numérica')
line_ex, = ax.plot([], [], '--', lw=2, label='Exacta')
ax.set_xlim(0, 1)
ax.set_ylim(-1.1, 1.1)
ax.set_xlabel('x')
ax.set_ylabel('u(x,t)')
ax.grid(alpha=0.25)
ax.legend()

for k in range(0, len(t_hist), max(1, len(t_hist)//12)):
    t = t_hist[k]
    line_num.set_data(x, U_hist[k])
    line_ex.set_data(x, exact_solution(x, t, c=a['c']))
    ax.set_title(f'Evolución temporal t={t:.3f}')
    plt.pause(0.08)

plt.show()


## Comentarios de cierre

- El esquema centrado en tiempo y espacio conserva la estructura oscilatoria de la ecuación de ondas.
- La condición CFL $\sigma\leq 1$ es clave para mantener estabilidad numérica.
- La malla temporal y espacial debe elegirse de forma consistente para equilibrar costo y precisión.
